**Install Dependencies**
- Install Litellm to enable the use of any vendor models

In [ ]:
# Install ADK and LiteLLM
!pip install google-adk -q
!pip install litellm -q

print("Google ADK and LiteLLM dependencies installed...")

Google ADK and LiteLLM dependencies installed...


**Configure environment**

In [27]:
import os
from getpass import getpass

# Get inputs
PROJECT_ID = getpass("Enter your GCP project id: ")
GOOGLE_MAPS_API_KEY = getpass("Enter your Google Maps API key: ")
GEMINI_API_KEY = getpass("Enter your Google Gemini API key: ")

# Set environment variables so LiteLLM and your functions automatically find them
os.environ["GOOGLE_MAPS_API_KEY"] = GOOGLE_MAPS_API_KEY
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

print("Credentials loaded successfully into environment!")

Enter your GCP project id: ··········
Enter your Google Maps API key: ··········
Enter your Google Gemini API key: ··········
Credentials loaded successfully into environment!


**Get NWS forecast based on coordinates**

In [18]:
from typing import Any, Dict, List, Union
import pandas as pd
import requests


def get_forecast_by_coordinates(
    latitude: float,
    longitude: float,
    user_agent: str = "GoogleColabNotebook/1.0 (user@example.com)",
    return_dataframe: bool = True,
    timeout: int = 10,
) -> Union[pd.DataFrame, List[Dict[str, Any]]]:
    """Retrieve weather forecast data from the NWS API for specific coordinates.

    Executes the two-stage NWS lookup process:
    1. Resolves grid points from `https://api.weather.gov/points/{lat},{lon}`.
    2. Retrieves period forecasts from the point's `forecast` property endpoint.

    Args:
        latitude (float): Latitude in decimal degrees (-90.0 to 90.0).
        longitude (float): Longitude in decimal degrees (-180.0 to 180.0).
        user_agent (str): Identification header required by NWS guidelines.
            Defaults to 'GoogleColabNotebook/1.0 (user@example.com)'.
        return_dataframe (bool): If True, returns a pandas DataFrame for neat
            display in Colab. If False, returns raw dictionary periods.
            Defaults to True.
        timeout (int): HTTP request timeout in seconds. Defaults to 10.

    Returns:
        Union[pd.DataFrame, List[Dict[str, Any]]]: A pandas DataFrame containing
            forecast periods if `return_dataframe=True`, or a raw list of
            dictionaries if False.

    Raises:
        ValueError: If coordinates are out of valid geographic ranges.
        requests.exceptions.HTTPError: If NWS API requests fail.
        requests.exceptions.RequestException: For network or connection errors.
    """
    # Validate coordinate ranges
    if not (-90.0 <= latitude <= 90.0):
        raise ValueError(f"Latitude must be between -90 and 90 degrees. Got {latitude}.")
    if not (-180.0 <= longitude <= 180.0):
        raise ValueError(f"Longitude must be between -180 and 180 degrees. Got {longitude}.")

    # Cap precision to 4 decimal places per NWS API recommendations
    lat_str = f"{latitude:.4f}"
    lon_str = f"{longitude:.4f}"

    headers = {
        "User-Agent": user_agent,
        "Accept": "application/geo+json",
    }

    # Step 1: Query points endpoint to get metadata and grid info
    points_url = f"https://api.weather.gov/points/{lat_str},{lon_str}"
    points_response = requests.get(points_url, headers=headers, timeout=timeout)
    points_response.raise_for_status()

    points_data = points_response.json()
    forecast_url = points_data.get("properties", {}).get("forecast")

    if not forecast_url:
        raise KeyError("Grid metadata response did not contain a valid 'forecast' URL.")

    # Step 2: Query the grid forecast endpoint
    forecast_response = requests.get(forecast_url, headers=headers, timeout=timeout)
    forecast_response.raise_for_status()

    forecast_data = forecast_response.json()
    periods: List[Dict[str, Any]] = forecast_data.get("properties", {}).get("periods", [])

    if return_dataframe:
        df = pd.DataFrame(periods)
        # Reorder key columns to the front for better visibility in Colab
        preferred_cols = ["name", "temperature", "temperatureUnit", "windSpeed", "windDirection", "shortForecast"]
        existing_cols = [col for col in preferred_cols if col in df.columns]
        other_cols = [col for col in df.columns if col not in existing_cols]

        return df[existing_cols + other_cols]

    return periods

In [19]:
# test get_forecast_by_coordinates function

# Fetch forecast as a Pandas DataFrame
df_forecast = get_forecast_by_coordinates(
    latitude=39.7456,
    longitude=-97.0892,
    user_agent="MyColabExperiment/1.0 (myemail@example.com)"
)

# Display table in Google Colab
df_forecast.head()

,name,temperature,temperatureUnit,windSpeed,windDirection,shortForecast,number,startTime,endTime,isDaytime,temperatureTrend,probabilityOfPrecipitation,icon,detailedForecast
0,Today,82,F,5 mph,E,Patchy Fog then Partly Sunny,1,2026-08-06T10:00:00-05:00,2026-08-06T18:00:00-05:00,True,None,"{'unitCode': 'wmoUnit:percent', 'value': 5}",https://api.weather.gov/icons/land/day/fog/bkn...,"Patchy fog before 11am. Partly sunny, with a h..."
1,Tonight,65,F,5 mph,SE,Chance Showers And Thunderstorms,2,2026-08-06T18:00:00-05:00,2026-08-07T06:00:00-05:00,False,None,"{'unitCode': 'wmoUnit:percent', 'value': 27}",https://api.weather.gov/icons/land/night/tsra_...,A chance of showers and thunderstorms after 11...
2,Friday,89,F,0 to 5 mph,S,Sunny,3,2026-08-07T06:00:00-05:00,2026-08-07T18:00:00-05:00,True,None,"{'unitCode': 'wmoUnit:percent', 'value': 10}",https://api.weather.gov/icons/land/day/few?siz...,"Sunny, with a high near 89. South wind 0 to 5 ..."
3,Friday Night,67,F,0 to 5 mph,E,Slight Chance Showers And Thunderstorms,4,2026-08-07T18:00:00-05:00,2026-08-08T06:00:00-05:00,False,None,"{'unitCode': 'wmoUnit:percent', 'value': 18}",https://api.weather.gov/icons/land/night/tsra_...,A slight chance of showers and thunderstorms b...
4,Saturday,90,F,5 to 15 mph,SE,Sunny,5,2026-08-08T06:00:00-05:00,2026-08-08T18:00:00-05:00,True,None,"{'unitCode': 'wmoUnit:percent', 'value': 7}",https://api.weather.gov/icons/land/day/few?siz...,"Sunny, with a high near 90. Southeast wind 5 t..."


**Function to get the lat/lon coordinates based on the city and state**

In [20]:
import os
from typing import Tuple
import requests


def get_coordinates(
    city: str,
    state: str,
    api_key: str = None,
    timeout: int = 10,
) -> Tuple[float, float]:
    """Convert a city and state into geographic latitude and longitude coordinates.

    Uses the Google Maps Geocoding API to resolve address strings to spatial coordinates.

    Args:
        city (str): The name of the city (e.g., 'Austin', 'Seattle').
        state (str): The state name or 2-letter postal abbreviation (e.g., 'TX', 'Washington').
        api_key (str, optional): Google Maps API Key. If not provided, reads from
            the 'GOOGLE_MAPS_API_KEY' environment variable.
        timeout (int): Request timeout in seconds. Defaults to 10.

    Returns:
        Tuple[float, float]: A tuple containing (latitude, longitude) as floats.

    Raises:
        ValueError: If an API key is missing or no results are returned for the input.
        requests.exceptions.HTTPError: If the Google API request fails.
    """
    key = api_key or os.getenv("GOOGLE_MAPS_API_KEY")
    if not key:
        raise ValueError(
            "Google Maps API Key required. Pass 'api_key' argument or set GOOGLE_MAPS_API_KEY env var."
        )

    address_str = f"{city.strip()}, {state.strip()}"
    base_url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {
        "address": address_str,
        "key": key,
    }

    response = requests.get(base_url, params=params, timeout=timeout)
    response.raise_for_status()

    data = response.json()
    status = data.get("status")

    if status != "OK" or not data.get("results"):
        error_msg = data.get("error_message", f"Geocoding API status: {status}")
        raise ValueError(f"Could not resolve coordinates for '{address_str}'. {error_msg}")

    location = data["results"][0]["geometry"]["location"]
    return location["lat"], location["lng"]

In [21]:
# test get_coordinates function

print(get_coordinates(city="Pittsburgh", state="PA", api_key=GOOGLE_MAPS_API_KEY))

(40.4386612, -79.99723519999999)


In [38]:
import json
import os
from litellm import completion # enables support for Gemini and other third-party models


# 1. System Instruction defining the agent's boundaries
SYSTEM_PROMPT = """
You are a helpful assistant specialized strictly in US weather forecasts using the National Weather Service (NWS).

Guidelines:
1. Scope: You only answer weather-related queries and questions about locations within the United States and its territories.
2. Out-of-Scope Requests: If a user asks a general question unrelated to weather (e.g., math, coding, general trivia, recipes), politely decline by stating: "I can only assist with weather-related inquiries for locations within the United States."
3. Foreign Locations: The NWS API only covers US cities and territories. If a location is outside the US (e.g., Paris, Tokyo, Toronto), explain that you can only provide weather forecasts for US locations.
"""


# 2. Define tool schemas for LiteLLM
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_coordinates",
            "description": "Converts a US city and state into geographic latitude and longitude coordinates.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, e.g. Pittsburgh"},
                    "state": {"type": "string", "description": "State name or abbreviation, e.g. PA"},
                },
                "required": ["city", "state"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_forecast_by_coordinates",
            "description": "Fetches current weather forecast from NWS using latitude and longitude.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {"type": "number", "description": "Latitude float coordinate"},
                    "longitude": {"type": "number", "description": "Longitude float coordinate"},
                },
                "required": ["latitude", "longitude"],
            },
        },
    },
]

# Map string tool names to actual python functions
AVAILABLE_FUNCTIONS = {
    "get_coordinates": get_coordinates,
    "get_forecast_by_coordinates": lambda latitude, longitude: get_forecast_by_coordinates(
        latitude=latitude,
        longitude=longitude,
        return_dataframe=False
    ),
}

def run_agent(user_prompt: str, model: str = "gemini-2.5-flash"):
    """Executes an agent loop supporting system instructions and tool evaluation via LiteLLM."""

    # Prepend system instruction to enforce assistant boundaries
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    while True:
        response = completion(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )

        response_message = response.choices[0].message

        # If no tool calls requested, the model handled it via text (e.g., declining off-topic prompts)
        if not response_message.tool_calls:
            return response_message.content

        messages.append(response_message.model_dump())

        for tool_call in response_message.tool_calls:
            func_name = tool_call.function.name

            if isinstance(tool_call.function.arguments, str):
                func_args = json.loads(tool_call.function.arguments)
            else:
                func_args = tool_call.function.arguments

            print(f"-> Agent calling function '{func_name}' with args: {func_args}")

            function_to_call = AVAILABLE_FUNCTIONS[func_name]

            try:
                result = function_to_call(**func_args)
                if func_name == "get_coordinates":
                    result = {"latitude": result[0], "longitude": result[1]}
            except Exception as err:
                # If a function fails (e.g., NWS returning 404 for non-US coordinates),
                # pass the error message back to the LLM gracefully.
                result = {"error": str(err)}

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id or func_name,
                    "name": func_name,
                    "content": json.dumps(result, default=str),
                }
            )


**Test code to ensure agent works with multiple US cities**

In [40]:
test_prompts = [
    "What is the weather forecast like in Portland, Maine right now?",
    "Can you tell me the weather in Tokyo, Japan?",
    "What is the weather forecast for Toronto, Canada?",
    "Can you write a Python script to sort a list?",
    "What is the capital of France?",
    "What is the weather forecast like in Fairbanks, Alaska right now?"
]

for prompt in test_prompts:
    print(f"🤖 Query: {prompt}")
    print("-" * 50)

    response = run_agent(prompt)
    print("\nAgent Answer:\n", response)
    print("\n" + "=" * 60 + "\n")

🤖 Query: What is the weather forecast like in Portland, Maine right now?
--------------------------------------------------
-> Agent calling function 'get_coordinates' with args: {'city': 'Portland', 'state': 'ME'}
-> Agent calling function 'get_forecast_by_coordinates' with args: {'longitude': -70.28438249999999, 'latitude': 43.670822}

Agent Answer:
 The weather in Portland, Maine this afternoon will be sunny with a high near 87 degrees Fahrenheit. The wind will be from the south at 5 to 10 mph.

Tonight, there will be patchy fog after 10 PM, with partly cloudy skies and a low around 69 degrees Fahrenheit. The southwest wind will be 0 to 10 mph.


🤖 Query: Can you tell me the weather in Tokyo, Japan?
--------------------------------------------------

Agent Answer:
 I can only provide weather forecasts for US locations.


🤖 Query: What is the weather forecast for Toronto, Canada?
--------------------------------------------------

Agent Answer:
 I can only provide weather forecasts f